# Repository edition: 05_pattern_experiments

Run from this repository in your existing GPU environment. Source is in `../suites/05_pattern_experiments/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Three Ways: faster code, fresh start on a new GPU

This notebook starts a NEW run on a new GPU instance using the faster curve evaluator. It does not require an old configuration or protocol file. It uses your existing environment and credentials; **no installation or login cells**.

**Starting here repeats the earlier computation.** If you want to preserve the old run instead, transfer its FULL results directory (including features, configuration.json and protocol.json), plus the original cohort input, and use Three_Ways_Fast_Resume.ipynb. A compact exported results ZIP is insufficient for a full resume.

The faster evaluator caches unchanged tokens and layers and validates each unfinished case against the original path, with fallback when checks fail. All experimental settings and fp32 precision are retained. No specific GH200 speedup is promised.

Includes pattern-fitted weight edits and fair live-gradient controls; gate/edge size curves and sign/magnitude tests; relation geometry; 1B/3B/8B/Qwen scaling; per-neuron log-factor ratios; OLMo 1B/7B final-success cohorts; prompt distance and sink sweeps.

**Full defaults:** all valid facts and discovered donor pairs, four inference models and two training trajectories. This is a large campaign that can take many GPU-days, not an overnight guarantee. fp32 8B weights alone take about 32 GB; plan for substantial additional GPU/CPU memory and hundreds of GB to over a TB of cache/checkpoint disk. An 80-GB-class GPU is a sensible planning target, not a measured guarantee. The full CUDA campaign has not been run here.

Tests use small randomly initialized Llama/Qwen/OLMo models; they are verification, not the experiment cohort.

Upload the **original `causal_v1_results.zip`** to this new GPU, or set INPUT_RESULTS to its full path. Do not use the compact gates/followup/pattern results ZIP as the cohort input. This fresh-start run writes to a new `patterns_fast_v1` folder.


## 1. Settings
`all` runs the complete campaign sequentially. You may run `capture`, then `analysis`, `weights`, `curves`, `sink`, `training`, and `report` separately with the **same configuration and run name**. Changing model lists, budgets or any protocol option requires a new run name.

To run only one model at full cohort size, change `MODEL_KEYS` before the first launch and use a distinct `RUN_NAME`. This does not reduce the number of facts or pairs for that model. Training has its own model list.


In [ ]:
from pathlib import Path
RUN_NAME = "patterns_fast_v1"
RUN_STAGE = "all"  # all, capture, analysis, weights, curves, sink, training, report
WORK = Path.home()
INPUT_RESULTS = None  # Auto-find original causal_v1 results; otherwise put its full path here.
PYTHON_OVERRIDE = None  # Existing environment's python path, if auto-detection is wrong.
MODEL_KEYS = ["llama1b", "llama3b", "llama8b", "qwen7b"]
TRAINING_MODEL_KEYS = ["olmo1b", "olmo7b"]
RUN_IN_BACKGROUND = True
OVERRIDES = {}  # Example: {"doses": [0.001, 0.003, 0.01, 0.03]}; defaults already run full experiments.


## 2. Prepare the repository source
This extracts the experiment code only. It does not install anything.


In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
BUNDLE = _source_bundle('05_pattern_experiments')
EXPECTED_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('05_pattern_experiments'))).hexdigest()
payload = base64.b64decode(BUNDLE)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_SHA256
CODE = WORK / ("threeways_pattern_fast_start_" + EXPECTED_SHA256[:12])
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for info in z.infolist():
        destination = (CODE / info.filename).resolve()
        if CODE.resolve() not in destination.parents:
            raise ValueError("Unsafe archive path")
        if destination.exists() and destination.read_bytes() != z.read(info):
            raise RuntimeError("Extracted source was changed. Use a fresh code directory to avoid mixing versions.")
    z.extractall(CODE)
print("Code:", CODE)


## 3. Locate your existing environment and input
The existing followup/causal environment is preferred to the notebook kernel if present. No credentials are requested. Model access must already be enabled for the requested repositories.


In [ ]:
candidates = [WORK / "threeways_followup_env/bin/python", WORK / "threeways_causal_env/bin/python", Path(sys.executable)]
PYTHON = Path(PYTHON_OVERRIDE).expanduser() if PYTHON_OVERRIDE else next(p for p in candidates if p.exists())
if INPUT_RESULTS:
    INPUT = Path(INPUT_RESULTS).expanduser().resolve()
else:
    candidates = [WORK / "causal_v1_results.zip", WORK / "threeways_causal_results/causal_v1", Path.cwd() / "causal_v1_results.zip"]
    INPUT = next((p for p in candidates if p.exists()), None)
if INPUT is None or not INPUT.exists():
    raise FileNotFoundError("Set INPUT_RESULTS to your ORIGINAL causal_v1_results.zip or extracted causal_v1 directory.")
if RUN_STAGE not in {"all", "capture", "analysis", "weights", "curves", "sink", "training", "report"}:
    raise ValueError("Unknown RUN_STAGE")
if Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be a simple folder name")
RESULTS = WORK / "threeways_pattern_results" / RUN_NAME
RESULTS.mkdir(parents=True, exist_ok=True)
# Read configuration with the selected environment, not the notebook kernel.
defaults = subprocess.check_output([str(PYTHON), "-c", "import json; from pattern_study.config import DEFAULT; print(json.dumps(DEFAULT))"], cwd=CODE, text=True)
CONFIG = json.loads(defaults)
known_models = {s["key"] for s in CONFIG["models"]}
known_training = {s["key"] for s in CONFIG["training_models"]}
if not set(MODEL_KEYS) <= known_models or not set(TRAINING_MODEL_KEYS) <= known_training:
    raise ValueError("Unknown model key")
CONFIG["models"] = [s for s in CONFIG["models"] if s["key"] in MODEL_KEYS]
CONFIG["training_models"] = [s for s in CONFIG["training_models"] if s["key"] in TRAINING_MODEL_KEYS]
CONFIG.update(OVERRIDES)
CONFIG_PATH = RESULTS / "configuration.json"
if (RESULTS / "protocol.json").exists() and json.loads((RESULTS / "protocol.json").read_text())["config"] != CONFIG:
    raise RuntimeError("Configuration changed. Choose a NEW RUN_NAME, or restore the previous configuration to resume.")
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print("Python:", PYTHON)
print("Input:", INPUT)
print("Results:", RESULTS)
print("Models:", [s["key"] for s in CONFIG["models"]])
print("Training:", [s["key"] for s in CONFIG["training_models"]])
print("Free disk GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## 4. Check the existing environment
This check imports libraries and reports GPU memory; it does not install packages. Full experiments use fp32 and eager attention. The 7B/8B runs cannot fit on 16/24-GB GPUs. No lower-precision fallback is silently applied.


In [ ]:
check = """import json, torch, transformers, numpy, scipy, pandas, sklearn, matplotlib, huggingface_hub
from pattern_study.config import validate
from pathlib import Path
import sys
cfg = json.loads(Path(sys.argv[1]).read_text())
validate(cfg)
if sys.argv[2] in ['all', 'training'] and cfg['training_models']:
    import datasets
assert transformers.__version__ == '4.48.3', 'Use your existing transformers 4.48.3 environment'
print('Torch:', torch.__version__, 'Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print('GPU:', torch.cuda.get_device_name(), 'free/total GiB:', round(free/2**30, 1), round(total/2**30, 1))
"""
subprocess.run([str(PYTHON), "-c", check, str(CONFIG_PATH), RUN_STAGE], cwd=CODE, check=True)


## 5. Launch or resume
The worker is detached from Jupyter, writes to disk, and survives a browser disconnect. Keep the Lambda instance running. It cannot survive instance termination or a machine reboot without being restarted.

The same run/config skips completed outputs. Do not launch another copy while it is active. A per-run filesystem lock also protects the results. `RUN_IN_BACKGROUND=True` returns immediately; use the monitor cell below.


In [ ]:
import time
LOG = RESULTS / "execution.log"
PID_FILE = RESULTS / "worker.pid"
STATUS = RESULTS / (RUN_STAGE + "_exit_code.txt")
if PID_FILE.exists():
    pid = int(PID_FILE.read_text())
    try:
        os.kill(pid, 0)
    except ProcessLookupError:
        PID_FILE.unlink()
    else:
        raise RuntimeError(f"Worker {pid} may still be running. Inspect the monitor before launching again.")
STATUS.unlink(missing_ok=True)
COMMAND = [str(PYTHON), "-u", "-m", "pattern_study.run", "--input", str(INPUT), "--out", str(RESULTS), "--config", str(CONFIG_PATH)]
if RUN_STAGE == "report":
    COMMAND += ["--report-only"]
elif RUN_STAGE != "all":
    COMMAND += ["--only", RUN_STAGE]
supervisor = """import os, pathlib, subprocess, sys, traceback
pid = pathlib.Path(sys.argv[1]); status = pathlib.Path(sys.argv[2]); pid.write_text(str(os.getpid()))
result = 1
try:
    result = subprocess.call(sys.argv[3:])
except BaseException:
    traceback.print_exc()
finally:
    status.write_text(str(result)); pid.unlink(missing_ok=True)
sys.exit(result)
"""
offset = LOG.stat().st_size if LOG.exists() else 0
with LOG.open("a") as log:
    worker = subprocess.Popen([str(PYTHON), "-u", "-c", supervisor, str(PID_FILE), str(STATUS), *COMMAND], cwd=CODE, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("Supervisor PID:", worker.pid)
print("Log:", LOG)
if not RUN_IN_BACKGROUND:
    try:
        with LOG.open() as log:
            log.seek(offset)
            while worker.poll() is None:
                text = log.read()
                if text:
                    print(text, end="", flush=True)
                time.sleep(1)
            print(log.read(), end="")
        if worker.returncode:
            raise RuntimeError(f"Worker exited with code {worker.returncode}; inspect {LOG}. Completed files are retained.")
    except KeyboardInterrupt:
        print("Monitoring stopped; the detached worker continues.")


## 6. Monitor progress
Run this cell whenever you return. Exit code `0` means the selected stage finished successfully. An absent exit code is not proof the process is alive; inspect the PID and log. A nonzero exit code means inspect the traceback before resuming.


In [ ]:
from collections import deque
if LOG.exists():
    with LOG.open() as f:
        print("".join(deque(f, maxlen=35)))
print("Exit code:", STATUS.read_text().strip() if STATUS.exists() else "not recorded")
if PID_FILE.exists():
    pid = int(PID_FILE.read_text())
    try:
        os.kill(pid, 0)
        print("Supervisor process exists:", pid)
    except ProcessLookupError:
        print("Recorded supervisor is no longer running:", pid)
else:
    print("No active supervisor PID recorded.")
print("Free disk GiB:", round(shutil.disk_usage(WORK).free / 2**30, 1))
print("Completed model stages:", [str(p.relative_to(RESULTS)) for p in RESULTS.glob("*/CORE_COMPLETE.json")])
print("Completed training runs:", [str(p.relative_to(RESULTS)) for p in RESULTS.glob("training_*/TRAINING_COMPLETE.json")])


## 7. Export compact results
Export after completion. For a deliberate partial snapshot set `ALLOW_PARTIAL=True`; the filename will say PARTIAL. Large feature caches and downloaded model weights remain on Lambda for resume/reanalysis. Downloading the compact ZIP alone does not preserve those caches.


In [ ]:
ALLOW_PARTIAL = False
complete = STATUS.exists() and STATUS.read_text().strip() == "0" and not PID_FILE.exists()
if not complete and not ALLOW_PARTIAL:
    raise RuntimeError("Selected stage has not completed. Check the monitor, or explicitly request a partial snapshot.")
ARCHIVE = WORK / (RUN_NAME + "_" + RUN_STAGE + ("_results.zip" if complete else "_PARTIAL.zip"))
excluded = {"features", "candidate_data", "__pycache__", ".pytest_cache"}
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULTS.rglob("*")):
        relative = p.relative_to(RESULTS)
        if p.is_file() and not excluded.intersection(relative.parts) and p.suffix != ".tmp":
            z.write(p, relative)
    z.write(CODE / "README.txt", "README.txt")
    z.write(CODE / "FAST_README.txt", "FAST_README.txt")
from IPython.display import FileLink, display
print("Archive:", ARCHIVE)
display(FileLink(str(ARCHIVE)))


## Interpretation notes before writing the paper

- A single input/output vector defines a rank-one fit. Higher-rank fitting includes independent calibration pairs. Attained ranks are saved; requested rank caps do not guarantee identical realized rank for each arm.
- Gate/up/down gradient arms share the live fixed-competitor margin objective; one-step directions are not equally converged optimizers.
- A successful edit does not uniquely locate the original cause or storage site. Same Frobenius norm does not ensure same functional cost: report specificity too.
- SiLU has no binary closed/open boundary. Sign interventions and magnitude interventions are distinct; some interventions can leave the native SiLU range.
- Cross-template attention uses explicit semantic-role mass transport, not claimed one-to-one token alignment or isolated Q/K edits.
- Pairwise cosine entries, paraphrase donors and random seeds are not independent facts. Final-success OLMo cohorts are retrospective and may still have few acquisition events.
- Only genuine training gradients can support the optional gradient-subspace analysis. It is skipped by default.

Detailed formulas, endpoint definitions, output filenames, limitations and official sources are in the embedded `README.txt`.

Official models: [Llama 3.2](https://huggingface.co/meta-llama/Llama-3.2-3B), [Llama 3.1](https://huggingface.co/meta-llama/Llama-3.1-8B), [Qwen 2.5](https://huggingface.co/Qwen/Qwen2.5-7B), [OLMo 2 7B](https://huggingface.co/allenai/OLMo-2-1124-7B). [OLMo checkpoint documentation](https://github.com/allenai/OLMo/blob/main/docs/Checkpoints.md).
